# Modelo base de ML para biomecánica de sprint

Este cuaderno sirve como punto de partida para crear un modelo de aprendizaje automático usando el dataset sintético del proyecto.

El objetivo actual es predecir:
- `time_10m_s`: tiempo de carrera de 10 metros.
- `quality_class`: clase de calidad (`buena`, `regular` o `mala`).

El dataset es sintético y sirve para validar el flujo de trabajo. Las métricas no representan resultados de atletas reales.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from training.features import DEFAULT_FEATURE_COLUMNS
from training.model import SprintMLConfig, SprintMLModel

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'drone_analysis').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data' / 'synthetic_ml'
MODEL_DIR = ROOT / 'models' / 'sprint_baseline'
TRIALS_PATH = DATA_DIR / 'trials.csv'
FRAMES_PATH = DATA_DIR / 'frame_features.csv'
ATHLETES_PATH = DATA_DIR / 'athletes.csv'
METADATA_PATH = DATA_DIR / 'dataset_metadata.json'

print(f'Raíz del proyecto: {ROOT}')
print(f'Dataset: {TRIALS_PATH}')

## 1. Cargar el dataset

In [ ]:
trials = pd.read_csv(TRIALS_PATH)
athletes = pd.read_csv(ATHLETES_PATH)
frames = pd.read_csv(FRAMES_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))

print('Atletas:', len(athletes))
print('Trials:', len(trials))
print('Frames:', len(frames))
print('Variables del dataset:', metadata['feature_columns'])

In [ ]:
trials.head(8)

## 2. Exploración básica

Revisamos la distribución de las variables objetivo y la calidad de las variables de entrada.

In [ ]:
display(trials['quality_class'].value_counts())
display(trials.groupby('split').size().rename('trials'))
trials[['time_10m_s', 'average_speed_ms', 'max_speed_ms', 'stride_length_m', 'cadence_steps_s', 'posture_score']].describe().round(3)

In [ ]:
trials[DEFAULT_FEATURE_COLUMNS].describe().T[['count', 'mean', 'std', 'min', 'max']].round(3)

## 3. Verificar separación por atleta

La separación debe hacerse por `athlete_id`, no por fila, para que un atleta no aparezca en entrenamiento y prueba.

In [ ]:
athlete_splits = trials.groupby('athlete_id')['split'].nunique()
assert (athlete_splits == 1).all(), 'Hay atletas en más de una partición'

split_athletes = trials.groupby('split')['athlete_id'].apply(set)
assert not (split_athletes['train'] & split_athletes['test'])
assert not (split_athletes['train'] & split_athletes['val'])
assert not (split_athletes['val'] & split_athletes['test'])
print('Sin fuga de datos entre partitioning de atletas.')

## 4. Entrenar el modelo base

Usamos `SprintMLModel`, que aplica imputación mediana y dos Random Forest: uno para regresión del tiempo y otro para clasificación de calidad.

In [ ]:
config = SprintMLConfig(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42,
)

model = SprintMLModel(config)
model.fit(trials)
model.metrics

## 5. Métricas del modelo

In [ ]:
metrics = model.metrics
print('Filas usadas:', metrics['rows'])
print('Atletas por partición:', metrics['athletes'])
print('Regresión:', metrics['regression'])
print('Clasificación:', metrics['classification'])

In [ ]:
test = trials[(trials['split'] == 'test') & (trials['quality_flag'] == 'ok')].copy()
test_predictions = model.predict(test).reset_index(drop=True)

comparison = test[['trial_id', 'athlete_id', 'time_10m_s', 'quality_class']].reset_index(drop=True)
comparison['predicted_time_10m_s'] = test_predictions['predicted_time_10m_s']
comparison['predicted_quality_class'] = test_predictions['predicted_quality_class']
comparison.head(10)

## 6. Importancia de variables

La importancia se calcula sobre el modelo de regresión del tiempo de 10 metros.

In [ ]:
importance = model.feature_importances().sort_values('regression_importance', ascending=True)
ax = importance['regression_importance'].plot(kind='barh', figsize=(10, 7), color='#2563eb')
ax.set_xlabel('Importancia')
ax.set_ylabel('Variable')
ax.set_title('Variables más importantes para predecir time_10m_s')
plt.tight_layout()
plt.show()

## 7. Predicción de un trial

La imputación permite predecir aunque una variable de confianza llegue como `NaN`.

In [ ]:
sample = trials.iloc[0].to_dict()
sample['pose_confidence'] = np.nan

prediction = model.predict_one(sample)
pd.DataFrame([{
    'predicted_time_10m_s': prediction['predicted_time_10m_s'],
    'predicted_quality_class': prediction['predicted_quality_class'],
}])

## 8. Preparar datos para un modelo temporal

`frame_features.csv` contiene la trayectoria por frame. Puede utilizarse posteriormente para LSTM, GRU o 1D-CNN sin cambiar el pipeline de captura.

In [ ]:
frame_summary = frames.groupby('trial_id').agg(
    frames=('frame_index', 'count'),
    duration_s=('time_s', 'max'),
    distance_m=('progress_m', 'max'),
    average_speed_ms=('speed_ms', 'mean'),
    average_cadence=('cadence_steps_s', 'mean'),
).reset_index()
frame_summary.head(10)

## 9. Siguientes pasos

1. Reemplazar `data/synthetic_ml/trials.csv` por/features extraídos de videos reales.
2. Mantener la partición por `athlete_id`.
3. Ajustar hiperparámetros con el split de validación.
4. Entrenar un modelo temporal usando `frame_features.csv`.
5. Comparar las métricas con la validación manual del proyecto.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / 'notebook_model.joblib')
print(f'Modelo guardado en {MODEL_DIR / "notebook_model.joblib"}')